In [19]:
import pandas as pd
# import jinja2

# CORGIS data
import electricity

# Custom module
import data_util
import plot_util

In [20]:
# Retrieve entire US-based dataset
utility = electricity.get_utility()
df = pd.json_normalize(utility)

# Limit to the state of New York and relevant columns
ny_df = data_util.get_state_data('NY', df)

# Add key metrics for use in later plots
ny_df = data_util.prepare_data(ny_df)

# list(ny_df.columns)

In [21]:
state_variance = data_util.get_state_variance(data_util.prepare_data(df))

top_variance_table = plot_util.get_state_variance_table(state_variance)
top_variance_table.show()

In [22]:
# Keep utilities that offer residential services
residential_customers_df = data_util.get_customer_utilities(
    ny_df,  "Residential").round(2)
res_boxplot = plot_util.get_utility_type_box_plot(
    residential_customers_df, "Residential")

res_boxplot.show()

# Keep utilities that offer industrial services
industrial_customers_df = data_util.get_customer_utilities(
    ny_df, "Industrial").round(2)
ind_boxplot = plot_util.get_utility_type_box_plot(
    industrial_customers_df, "Industrial")

ind_boxplot.show()

In [23]:
# Key metrics: System Loss %, Load Factor', Industrial Revenue %, Price Spread, FairnessIndex
heatmap = plot_util.get_key_metrics_corr_matrix(ny_df)

heatmap.show()

In [24]:
# Keep residential utilities that are using energy (instead of ONLY reseale, etc.)
residential_lf_sys_loss_df = data_util.get_residential_sys_loss(ny_df)
residential_lf_sys_loss_df = data_util.get_residential_load_factor(
    residential_lf_sys_loss_df).round(2)

dual_y_scatter = plot_util.get_fairness_dual_y_scatter_plot(
    residential_lf_sys_loss_df)

dual_y_scatter.show()

In [25]:
# Keep utilities that offer both industrial and residential services
res_ind_customers_df = data_util.get_customer_utilities(
    ny_df, "Industrial").round(2)

dumbbell = plot_util.get_rate_disparity_dumbbell_plot(res_ind_customers_df)

dumbbell.show()

In [26]:
# Look at a specific utility's energy usage/flow
utility_usage = data_util.get_utility_usage(ny_df.iloc[10])
sankey = plot_util.get_energy_use_sankey_plot(utility_usage)

sankey.show()

# Look at the entire country's energy usage/flow
us_energy_flow = df.groupby(["Utility.State"]).sum().sum()[1:]
us_energy_flow = data_util.get_utility_usage(us_energy_flow, level="US")

us_sankey = plot_util.get_energy_use_sankey_plot(us_energy_flow)

us_sankey.show()

In [27]:
# Gather all plots and export them as SVGs
plots = [top_variance_table, res_boxplot, ind_boxplot, heatmap,
         dual_y_scatter, dumbbell, sankey, us_sankey]

plot_util.export_plots_as_svg(plots)